In [ ]:
from pathlib import Path
import pandas as pd

datapack_2021 = Path("../data/2021_GCP_SA3_for_AUS_short-header")

# check BEFORE the loop, so a wrong path fails loudly
if not datapack_2021.exists():
    raise FileNotFoundError(f"Not found: {datapack_2021.resolve()}")

files = sorted(datapack_2021.rglob("2021Census_G*_SA3.csv"))
print(f"{len(files)} SA3 files found in total")

for p in files:
    if p.name.split("_")[1][:3] in ("G09", "G11", "G13"):
        cols = pd.read_csv(p, nrows=0).columns[1:]    # nrows=0 reads the header only
        print(f"{p.name:35} {len(cols):4} cols   {cols[0]}  ...  {cols[-1]}")

In [ ]:
from pathlib import Path

datapack_2021 = Path("../data/2021_GCP_SA3_for_AUS_short-header")
print(datapack_2021.resolve(), "exists:", datapack_2021.exists())

In [ ]:
def cols_of(table_part):
    """Header names of one DataPack part, e.g. 'G09C'."""
    p = next(datapack_2021.rglob(f"2021Census_{table_part}_AUST_SA3.csv"))
    return pd.read_csv(p, nrows=0).columns[1:].tolist()

# G09: one female column per birthplace category (using the 25-44 band as a sample)
g09_f = [c for part in "CDEF" for c in cols_of(f"G09{part}")
         if c.startswith("F_") and c.endswith("_25_44")]
print(f"G09 female birthplace categories ({len(g09_f)}):")
print([c.removeprefix("F_").removesuffix("_25_44") for c in g09_f])

# G13: female columns that look like totals
g13_f = [c for part in "BC" for c in cols_of(f"G13{part}")
         if c.startswith("F") and "Tot" in c]
print(f"\nG13 female total-type columns ({len(g13_f)}):")
print(g13_f)

# G11: all age prefixes, plus the total columns
g11 = [c for part in "ABCD" for c in cols_of(f"G11{part}")]
print("\nG11 age prefixes:", sorted({c.split("_")[0] for c in g11}))
print("G11 total columns:", [c for c in g11 if c.startswith("T_")])

In [ ]:
g11_all = set(c for part in "ABCD" for c in cols_of(f"G11{part}"))
wanted = [f"A{b}_{cat}_T" for b in ["25_34","35_44","45_54","55_64","65_74"]
          for cat in ["SEO","UOLSE_VWW","UOLSE_NWNAA","T"]]
print("missing:", [c for c in wanted if c not in g11_all])

In [ ]:
!python ../scripts/Census_data_extraction.py extract

In [ ]:
import pandas as pd

df = pd.read_csv("../data/extracted/census_2021_sa3_long.csv", dtype={"geo_code": str})
w = df.pivot(index="geo_code", columns="source_column", values="value")   # one row per SA3

# 1. Completeness: expect (340, 39), no gaps, no negatives
print("shape:", w.shape, "| missing:", w.isna().sum().sum(), "| negatives:", (w < 0).sum().sum())

# 2. Internal consistency (small gaps are normal: ABS randomly adjusts cells for privacy)
g13_parts = w["FSEO_Tot"] + w["FOL_Tot_Tot"] + w["F_LUatH_NS_Tot"]
print("\nG13 parts / total, should be ~1.00:")
print((g13_parts / w["F_Tot_Tot"]).describe().round(3))

bands = ["25_44", "45_54", "55_64", "65_74"]
over = sum((w[f"F_Australia_{b}"] + w[f"F_COB_NS_{b}"] > w[f"F_Tot_{b}"]).sum() for b in bands)
print("G09 cells where Australia + not stated > total:", over)

# 3. The three measures (same formulas as the SQL design)
tot = sum(w[f"F_Tot_{b}"] for b in bands)
aus = sum(w[f"F_Australia_{b}"] for b in bands)
ns  = sum(w[f"F_COB_NS_{b}"] for b in bands)
m = pd.DataFrame({
    "pct_born_overseas":   100 * (tot - aus - ns) / (tot - ns),
    "pct_other_language":  100 * w["FOL_Tot_Tot"] / (w["FSEO_Tot"] + w["FOL_Tot_Tot"]),
    "pct_low_proficiency": 100 * w["F_Tot_UOLSE_NWorNAA"]
                           / (w["FSEO_Tot"] + w["F_Tot_UOLSE_VWorW"] + w["F_Tot_UOLSE_NWorNAA"]),
    "pct_birthplace_not_stated": 100 * ns / tot,
    "women_25_74": tot,
})
print("\nMeasure distributions:")
print(m.describe(percentiles=[.05, .5, .95]).round(1))

# 4. Areas where 'not stated' is high enough to make shares shaky
print("\nAreas with >10% birthplace not stated:", (m["pct_birthplace_not_stated"] > 10).sum())
print("Areas with <1,000 women aged 25-74:", (m["women_25_74"] < 1000).sum())

In [ ]:
!python ../scripts/load_census_data_to_db.py check
!python ../scripts/load_census_data_to_db.py load

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("../data")
FOLDERS = {
    2011: DATA / "2011_BCP_SA3_for_AUST_short-header",
    2016: DATA / "2016_GCP_SA3_for_AUS_short-header",
    2021: DATA / "2021_GCP_SA3_for_AUS_short-header",
}

def sa3_codes(folder):
    """Find the first SA3 CSV in a DataPack folder and return its SA3 codes (first column)."""
    if not folder.exists():
        raise FileNotFoundError(f"Not found: {folder.resolve()}")
    files = sorted(folder.rglob("*SA3*.csv"))
    if not files:
        raise FileNotFoundError(f"No SA3 CSV files under {folder.resolve()}")
    df = pd.read_csv(files[0], dtype=str, usecols=[0])
    print(f"{folder.name}: using {files[0].name}, code column '{df.columns[0]}', {len(df)} areas")
    return set(df.iloc[:, 0])

codes = {year: sa3_codes(folder) for year, folder in FOLDERS.items()}

def non_spatial(c):                     # x9797 migratory/offshore, x9999 no usual address
    return c.endswith(("9797", "9999"))

for a, b in [(2011, 2016), (2016, 2021)]:
    only_a = sorted(c for c in codes[a] - codes[b] if not non_spatial(c))
    only_b = sorted(c for c in codes[b] - codes[a] if not non_spatial(c))
    print(f"\n{a} vs {b}: shared {len(codes[a] & codes[b])}")
    print(f"  only in {a} (excluding non-spatial): {only_a}")
    print(f"  only in {b} (excluding non-spatial): {only_b}")

In [ ]:
def table_files(folder):
    """Map table id (e.g. 'B09A', 'G13C') -> file, for every SA3 CSV in a DataPack folder."""
    return {p.name.split("_")[1]: p for p in sorted(folder.rglob("*SA3*.csv"))}

for year in (2011, 2016):
    print(f"\n===== {year} =====")
    files = table_files(FOLDERS[year])
    for tid, path in files.items():
        if tid[1:3] in ("08", "09", "10", "11", "12", "13", "14"):    # the cultural-diversity range
            cols = pd.read_csv(path, nrows=0).columns[1:]
            print(f"{tid:6} {len(cols):4} cols   {cols[0]}  ...  {cols[-1]}")

    # Birthplace table: female categories and age bands (uses the 25-34 / 25-44 band as a sample)
    birth = [c for tid, p in files.items() if tid[1:3] == "09"
             for c in pd.read_csv(p, nrows=0).columns[1:]]
    print("\nFemale birthplace columns ending in a 25-xx band:",
          [c for c in birth if c.startswith("F_") and ("_25_" in c)][:8], "...")
    print("Age bands (from F_Tot_*):", [c for c in birth if c.startswith("F_Tot_")])

    # Language table: female total-type columns
    lang = [c for tid, p in files.items() if tid[1:3] == "13"
            for c in pd.read_csv(p, nrows=0).columns[1:]]
    print("\nFemale language total columns:", [c for c in lang if c.startswith("F") and "Tot" in c][-12:])

In [ ]:
f11 = table_files(FOLDERS[2011])
f16 = table_files(FOLDERS[2016])
cols = lambda files, prefix: [c for tid, p in files.items() if tid.startswith(prefix)
                              for c in pd.read_csv(p, nrows=0).columns[1:]]

# 2011 birthplace (B09): Australia, total and not-stated columns, all sexes, to see the naming
b09 = cols(f11, "B09")
print("2011 B09:", [c for c in b09 if any(k in c for k in ("Australia", "Tot", "NS", "stated"))])

# 2011 language (B13): every column that isn't a single named language
b13 = cols(f11, "B13")
print("\n2011 B13 total/summary columns:",
      [c for c in b13 if any(k in c for k in ("SEO", "Tot", "Total", "NS", "stated", "well", "Well"))])
print("2011 B13 sample of all columns:", b13[:12])

# 2016: confirm the names the mapping will use
g09 = cols(f16, "G09"); g13 = cols(f16, "G13")
wanted16 = [f"F_{c}_{b}" for c in ("Tot", "Australia", "COB_NS")
            for b in ("25_44", "45_54", "55_64", "65_74", "Tot")] + ["FSEO_Tot", "FOL_Tot_Tot", "F_LSatH_NS_Tot"]
print("\n2016 missing:", [c for c in wanted16 if c not in g09 + g13])

In [ ]:
!python ../scripts/Census_data_extraction.py extract --year 2021
!python ../scripts/Census_data_extraction.py extract --year 2016
!python ../scripts/Census_data_extraction.py extract --year 2011


In [ ]:
!python ../scripts/load_census_data_to_db.py load
!python ../scripts/load_census_data_to_db.py load --csv ../data/extracted/census_2016_sa3_long.csv
!python ../scripts/load_census_data_to_db.py load --csv ../data/extracted/census_2011_sa3_long.csv

In [ ]:
import os, pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
load_dotenv("../scripts/.env")
assert os.getenv("DATABASE_URL"), "DATABASE_URL still not found: check the path to .env"
url = os.getenv("DATABASE_URL").replace("postgres://", "postgresql://", 1)
engine = create_engine(url.replace("postgresql://", "postgresql+psycopg2://", 1))

trend = pd.read_sql(text("SELECT * FROM v_sa3_census_trend"), engine)
for col in ["pct_born_overseas", "pct_other_language", "females_all_ages"]:
    trend[col] = trend[col].astype(float)

born = trend.pivot(index="geo_code", columns="census_year", values="pct_born_overseas")
print("2016 vs 2021 born overseas, Spearman:", round(born[2016].corr(born[2021], method="spearman"), 3))

pop = trend.pivot(index="geo_code", columns="census_year", values="females_all_ages")
change = (pop[2021] / pop[2011] - 1) * 100
print("Areas with female population change beyond +/-30%:", (change.abs() > 30).sum())
print(change.abs().sort_values(ascending=False).head(10).round(1))

In [ ]:
names = trend.drop_duplicates("geo_code").set_index("geo_code")["geo_name"]
big = pd.DataFrame({"name": names, "f_2011": pop[2011], "f_2021": pop[2021],
                    "change_pct": change.round(1)})
print(big[big["change_pct"].abs() > 30].sort_values("change_pct").to_string())

In [ ]:
g09_2021 = [c for p in sorted(FOLDERS[2021].rglob("2021Census_G09*_SA3.csv"))
            for c in pd.read_csv(p, nrows=0).columns[1:]]
cats = [c.removeprefix("F_").removesuffix("_25_44") for c in g09_2021
        if c.startswith("F_") and c.endswith("_25_44")]
print(len(cats), "categories:", cats)